<a href="https://colab.research.google.com/github/YB441/codsoft/blob/main/creadit_card_fraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the training data
train_df = pd.read_csv('/content/fraudTrain.csv')

# Display data types and column names
print("Training Data Info:")
train_df.info()

## 1. Setup and Data Loading

First, let's load the necessary libraries and your training and testing datasets (`fraudTrain.csv` and `fraudTest.csv`).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import matplotlib.pyplot as plt
import seaborn as sns

# Load the test data
test_df = pd.read_csv('/content/fraudTest.csv')

print("Test Data Info:")
test_df.info()

## 2. Initial Data Cleaning and Feature Engineering

We need to perform some initial cleaning on both datasets. This includes:
- Dropping the 'Unnamed: 0' column which is just an index.
- Handling missing values by dropping rows where the target `is_fraud` or critical numerical features (`unix_time`, `merch_lat`, `merch_long`) are missing. Given the small number of missing values (1 in training, if any in test), dropping is a safe approach.
- Converting `trans_date_trans_time` to datetime objects.
- Creating new time-based features from `trans_date_trans_time` to capture potential temporal patterns (e.g., hour of day, day of week).
- Dropping `cc_num` and `trans_num` as they are likely identifiers and not useful as direct features. `first`, `last`, `street`, `city`, `state`, `zip`, `dob`, `merchant`, `category`, `job` are categorical and will be handled during preprocessing.

In [ ]:
def preprocess_data(df):
    # Drop 'Unnamed: 0' column if it exists
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    # Drop rows with any missing values in critical columns
    # Identified missing values in 'is_fraud', 'unix_time', 'merch_lat', 'merch_long' from train_df.info()
    # Applying to both for consistency.
    df = df.dropna(subset=['is_fraud', 'unix_time', 'merch_lat', 'merch_long'])

    # Convert 'trans_date_trans_time' to datetime
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

    # Extract time-based features
    df['hour'] = df['trans_date_trans_time'].dt.hour
    df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['month'] = df['trans_date_trans_time'].dt.month
    df['day_of_year'] = df['trans_date_trans_time'].dt.dayofyear # Could be useful for seasonality

    # Drop original datetime column and identifier columns
    df = df.drop(columns=['trans_date_trans_time', 'cc_num', 'trans_num', 'first', 'last', 'dob'])

    return df

# Apply preprocessing to both training and testing datasets
train_df_processed = preprocess_data(train_df.copy())
test_df_processed = preprocess_data(test_df.copy())

print("\nTrain Data after preprocessing:")
print(train_df_processed.head())
print("\nTest Data after preprocessing:")
print(test_df_processed.head())

## 3. Data Exploration & Target Analysis

Now, let's examine the class distribution of the target variable `is_fraud` in the training data to understand the imbalance. We will also define our features (X) and target (y) for both datasets.

In [ ]:
# Class distribution in training data
print("\nClass distribution of 'is_fraud' in training data:")
fraud_counts = train_df_processed['is_fraud'].value_counts()
print(fraud_counts)
print(f"Legitimate transactions: {fraud_counts[0]:.0f} ({fraud_counts[0]/len(train_df_processed)*100:.2f}%)")
print(f"Fraudulent transactions: {fraud_counts[1]:.0f} ({fraud_counts[1]/len(train_df_processed)*100:.2f}%)")

# Define features (X) and target (y) for training and testing sets
X_train = train_df_processed.drop(columns=['is_fraud'])
y_train = train_df_processed['is_fraud']

X_test = test_df_processed.drop(columns=['is_fraud'])
y_test = test_df_processed['is_fraud']

print(f"\nX_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

## 4. Feature Preprocessing Pipeline

We will now set up a robust preprocessing pipeline using `ColumnTransformer` and `Pipeline`. This will handle:
- **Numerical Features**: `amt` and `unix_time` will be scaled using `RobustScaler` as specified, due to their potential for outliers. Other numerical features (`lat`, `long`, `city_pop`, `merch_lat`, `merch_long`, `hour`, `day_of_week`, `month`, `day_of_year`) will be scaled using `StandardScaler`.
- **Categorical Features**: Features like `merchant`, `category`, `gender`, `street`, `city`, `state`, `zip`, `job` will be one-hot encoded.

This preprocessing will be applied consistently to both training and testing data.

In [ ]:
# Identify numerical and categorical features
# Exclude 'is_fraud' which is the target
numerical_features = ['amt', 'unix_time', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'hour', 'day_of_week', 'month', 'day_of_year']
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

# Define preprocessing steps within ColumnTransformer directly
preprocessor = ColumnTransformer(transformers=[
    ('robust_scaler_amt_time', RobustScaler(), ['amt', 'unix_time']),
    ('standard_scaler_others', StandardScaler(), [col for col in numerical_features if col not in ['amt', 'unix_time']]),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_features)
], remainder='passthrough')

print("Preprocessing pipeline created.")

## 5. Advanced Imbalance Handling & Model Training

We will implement three classifiers and handle the class imbalance using two strategies:

1.  **SMOTE (Synthetic Minority Over-sampling Technique)**: This will generate synthetic samples for the minority class in the training data.
2.  **Class Weights**: This strategy assigns higher weights to the minority class during model training.

We'll train each model type with both imbalance handling approaches and evaluate them on the *untouched, raw, imbalanced test set*.

In [ ]:
# --- Models to compare ---
models = {
    'Logistic Regression': LogisticRegression(solver='liblinear', random_state=42, n_jobs=-1, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1)
}

# Store results
results = {}

# --- Evaluation Function ---
def evaluate_model(model_name, y_true, y_pred, y_prob):
    print(f"\n--- {model_name} Evaluation ---")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix (True vs Predicted):")
    cm = confusion_matrix(y_true, y_pred)
    print(pd.DataFrame(cm, index=['Actual Negative', 'Actual Positive'], columns=['Predicted Negative', 'Predicted Positive']))
    print(f"  False Negatives (Missed Fraud): {cm[1, 0]}")
    print(f"  False Positives (Blocked Legitimate): {cm[0, 1]}")
    print(f"\nROC-AUC Score: {roc_auc_score(y_true, y_prob):.4f}")


# --- Training with SMOTE ---
print("\n### Training Models with SMOTE ###")
for name, model in models.items():
    print(f"\nTraining {name} with SMOTE...")
    smote_pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)), # Removed n_jobs
        ('classifier', model)
    ])
    smote_pipeline.fit(X_train, y_train)
    y_pred = smote_pipeline.predict(X_test)
    y_prob = smote_pipeline.predict_proba(X_test)[:, 1]
    results[f'{name} (SMOTE)'] = {'y_pred': y_pred, 'y_prob': y_prob}
    evaluate_model(f'{name} (SMOTE)', y_test, y_pred, y_prob)


# --- Training with Class Weights ---
print("\n### Training Models with Class Weights ###")
# Calculate class weights for 'balanced' option
# This is done based on the original imbalanced training set
class_weights = len(y_train) / (2 * np.bincount(y_train.astype(int)))
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

for name, model in models.items():
    print(f"\nTraining {name} with Class Weights...")
    # Clone model to avoid modifying SMOTE-trained instance
    model_cw = model.__class__(**model.get_params())

    if 'class_weight' in model_cw.get_params():
        model_cw.set_params(class_weight='balanced')

    cw_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model_cw)
    ])
    cw_pipeline.fit(X_train, y_train)
    y_pred = cw_pipeline.predict(X_test)
    y_prob = cw_pipeline.predict_proba(X_test)[:, 1]
    results[f'{name} (Class Weights)'] = {'y_pred': y_pred, 'y_prob': y_prob}
    evaluate_model(f'{name} (Class Weights)', y_test, y_pred, y_prob)

## 6. Top Critical Features for Random Forest

Let's extract the top 5 most critical features from the Random Forest model trained using class weights (as it often provides more stable feature importances without synthetic data). This will give insights into what drives fraud detection.

In [ ]:
# Re-train Random Forest with class weights to get feature importances
# Ensure we have the fitted pipeline
rf_cw_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'))
])
rf_cw_pipeline.fit(X_train, y_train)

# Get feature importances from the Random Forest classifier
rf_classifier = rf_cw_pipeline.named_steps['classifier']
feature_importances = rf_classifier.feature_importances_

# Get feature names from the fitted preprocessor
feature_names = rf_cw_pipeline.named_steps['preprocessor'].get_feature_names_out()

# Create a Series for easier sorting
importance_series = pd.Series(feature_importances, index=feature_names)

# Get top 5 features
top_5_features = importance_series.nlargest(5)

print("\nTop 5 Most Critical Features (Random Forest with Class Weights):")
print(top_5_features)

# Optional: Visualize feature importances
fig = plt.figure(figsize=(10, 6))
sns.barplot(x=top_5_features.values, y=top_5_features.index, palette='viridis')
plt.title('Top 5 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning for Random Forest (Recall Optimization)

We will use `RandomizedSearchCV` to find the best hyperparameters for our `RandomForestClassifier` with `class_weight='balanced'`. The primary goal is to optimize for **recall** of the minority class (fraudulent transactions). We'll search across parameters like `n_estimators`, `max_features`, `max_depth`, `min_samples_split`, and `min_samples_leaf`.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Define the Random Forest model with class weights
rf_classifier_tuned = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced')

# Create a pipeline with preprocessing and the classifier
tuning_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf_classifier_tuned)
])

# Define the parameter distribution for RandomizedSearchCV
# Using distributions for faster and more effective search
param_dist = {
    'classifier__n_estimators': randint(50, 200), # Number of trees in the forest
    'classifier__max_features': ['sqrt', 'log2', None], # Number of features to consider when looking for the best split
    'classifier__max_depth': randint(5, 20), # Maximum number of levels in tree
    'classifier__min_samples_split': randint(2, 15), # Minimum number of samples required to split an internal node
    'classifier__min_samples_leaf': randint(1, 10), # Minimum number of samples required to be at a leaf node
}

# Set up RandomizedSearchCV to optimize for recall
# We use 'recall' as the scoring metric to prioritize minimizing False Negatives
random_search = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=param_dist,
    n_iter=50, # Number of parameter settings that are sampled. Adjust based on computational budget.
    scoring='recall', # Optimize for recall of the positive class
    cv=3, # Number of cross-validation folds
    verbose=2,
    random_state=42,
    n_jobs=-1 # Use all available cores
)

print("Starting Randomized Search for Random Forest hyperparameters...")
random_search.fit(X_train, y_train)

print(f"\nBest parameters found: {random_search.best_params_}")
print(f"Best recall score found: {random_search.best_score_:.4f}")

Starting Randomized Search for Random Forest hyperparameters...
Fitting 3 folds for each of 50 candidates, totalling 150 fits


## 8. Re-evaluating Random Forest with Tuned Hyperparameters

Now that we have the best hyperparameters from `RandomizedSearchCV`, let's evaluate the performance of the Random Forest model with these optimized settings on our untouched test set. We will use the same risk-averse evaluation metrics.

In [ ]:
print("\n--- Evaluating Tuned Random Forest (Class Weights) ---")

best_rf_model = random_search.best_estimator_
y_pred_tuned_rf = best_rf_model.predict(X_test)
y_prob_tuned_rf = best_rf_model.predict_proba(X_test)[:, 1]

# Evaluate the tuned model using our custom function
evaluate_model(
    'Tuned Random Forest (Class Weights)',
    y_test,
    y_pred_tuned_rf,
    y_prob_tuned_rf
)

# Extract and list top 5 most critical features from the tuned model
# This assumes the best_estimator_ is a pipeline and has a 'classifier' step
if hasattr(best_rf_model.named_steps['classifier'], 'feature_importances_'):
    tuned_feature_importances = best_rf_model.named_steps['classifier'].feature_importances_
    tuned_feature_names = best_rf_model.named_steps['preprocessor'].get_feature_names_out()

    tuned_importance_series = pd.Series(tuned_feature_importances, index=tuned_feature_names)
    tuned_top_5_features = tuned_importance_series.nlargest(5)

    print("\nTop 5 Most Critical Features (Tuned Random Forest):")
    print(tuned_top_5_features)

    # Optional: Visualize feature importances for the tuned model
    fig = plt.figure(figsize=(10, 6))
    sns.barplot(x=tuned_top_5_features.values, y=tuned_top_5_features.index, palette='viridis')
    plt.title('Top 5 Feature Importances (Tuned Random Forest)')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()